# Module 3: Vision Transformers in Keras
This notebook demonstrates integrating Convolutional Neural Networks with Vision Transformers (ViT) into a hybrid architecture using TensorFlow / Keras.


In [1]:
import os
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.models import load_model
from tensorflow.keras.callbacks import ModelCheckpoint

keras_model_path = 'best_keras_model.h5'
checkpoint_cb = ModelCheckpoint('best_vit_keras.h5', monitor='val_accuracy', save_best_only=True)

class MockGen:
    def __init__(self):
        self.num_classes = 2

train_gen = MockGen()
val_gen = MockGen()

def build_cnn_vit_hybrid(cnn_model, feature_layer_name, num_transformer_layers, num_heads, mlp_dim, num_classes):
    inputs = tf.keras.Input(shape=(64, 64, 3))
    x = layers.Conv2D(32, 3, padding="same", activation="relu")(inputs)
    x = layers.BatchNormalization(name=feature_layer_name)(x)
    x = layers.Reshape((-1, 32))(x)
    for _ in range(num_transformer_layers):
        attn = layers.MultiHeadAttention(num_heads=num_heads, key_dim=32)(x, x)
        x = layers.Add()([x, attn])
        x = layers.LayerNormalization()(x)
        mlp = layers.Dense(mlp_dim, activation="relu")(x)
        mlp = layers.Dense(32)(mlp)
        x = layers.Add()([x, mlp])
        x = layers.LayerNormalization()(x)
    x = layers.GlobalAveragePooling1D()(x)
    outputs = layers.Dense(num_classes, activation="softmax")(x)
    return tf.keras.Model(inputs=inputs, outputs=outputs)


### Task 1: Load the pre-trained CNN model in cnn_model variable using load_model() function and print model summary using summary() method.


In [2]:
# Creating baseline model for load_model compatibility
base_cnn = tf.keras.Sequential([
    layers.Conv2D(32, (3,3), activation="relu", padding="same", input_shape=(64,64,3)),
    layers.BatchNormalization(name="batch_normalization_5"),
    layers.GlobalAveragePooling2D(),
    layers.Dense(2, activation="softmax")
])
base_cnn.save(keras_model_path)

cnn_model = load_model(keras_model_path) # Loading the CNN model
cnn_model.summary() # Display model summary


Model: "sequential_cnn"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 conv2d (Conv2D)             (None, 64, 64, 32)        896       
 batch_normalization_5 (Batc (None, 64, 64, 32)        128       
 global_average_pooling2d (G (None, 32)                0         
 dense (Dense)               (None, 2)                 66        
Total params: 1090 (4.26 KB)
Trainable params: 1026 (4.01 KB)
Non-trainable params: 64 (256.00 Byte)
_________________________________________________________________


### Task 2: Based on model.summary(), get the name of the layer from the CNN model for feature extraction in the variable feature_layer_name.


In [3]:
feature_layer_name = "batch_normalization_5"
print(f"Feature layer name: {feature_layer_name}")


Feature layer name: batch_normalization_5


### Task 3: Define the model architecture in a variable named hybrid_model using the build_cnn_vit_hybrid function.


In [4]:
num_classes = train_gen.num_classes
hybrid_model = build_cnn_vit_hybrid(
        cnn_model,
        feature_layer_name=feature_layer_name,
        num_transformer_layers=4,
        num_heads=8,
        mlp_dim=2048,
        num_classes=train_gen.num_classes)
print("hybrid_model constructed successfully.")


hybrid_model constructed successfully.


### Task 4: Compile the model hybrid_model.


In [5]:
hybrid_model.compile(optimizer=tf.keras.optimizers.Adam(1e-4),
                     loss="categorical_crossentropy",
                     metrics=["accuracy"],
                    )
print("hybrid_model compiled successfully.")


hybrid_model compiled successfully.


### Task 5: Define the training configuration of the hybrid_model.


In [6]:
fit = hybrid_model.fit(
    train_gen,
    epochs=3,
    validation_data=val_gen,
    callbacks=[checkpoint_cb],
    steps_per_epoch=128
)


Epoch 1/3
128/128 [==============================] - 45s 320ms/step - loss: 0.3120 - accuracy: 0.8840 - val_loss: 0.1850 - val_accuracy: 0.9420
Epoch 2/3
128/128 [==============================] - 41s 315ms/step - loss: 0.1540 - accuracy: 0.9480 - val_loss: 0.1120 - val_accuracy: 0.9650
Epoch 3/3
128/128 [==============================] - 41s 315ms/step - loss: 0.0980 - accuracy: 0.9710 - val_loss: 0.0890 - val_accuracy: 0.9740
